In [5]:
import os
import json
import shutil
from pathlib import Path

def prepare_yolo_dataset(src_root, dest_root):
    """
    Reorganizes COCO format dataset into YOLO directory structure and creates YAML file
    
    Args:
        src_root (str): Path to your current dataset (data_twowheeler)
        dest_root (str): Where to create the new YOLO-formatted dataset
    """
    # Create destination directory structure
    splits = ['train', 'valid', 'test']
    dest_paths = {}
    
    for split in splits:
        # Create image and label directories
        dest_paths[split] = {
            'images': os.path.join(dest_root, split, 'images'),
            'labels': os.path.join(dest_root, split, 'labels')
        }
        os.makedirs(dest_paths[split]['images'], exist_ok=True)
        os.makedirs(dest_paths[split]['labels'], exist_ok=True)
    
    # Process each split
    class_names = set()
    
    for split in splits:
        src_split_path = os.path.join(src_root, split)
        anno_file = os.path.join(src_split_path, '_annotations.json')
        
        if not os.path.exists(anno_file):
            print(f"Skipping {split} - annotations.json not found")
            continue
        
        with open(anno_file) as f:
            data = json.load(f)
        
        # Create mappings
        id_to_image = {img['id']: img for img in data['images']}
        categories = {cat['id']: cat['name'] for cat in data.get('categories', [])}
        
        # Process annotations
        for ann in data['annotations']:
            img_id = ann['image_id']
            img_info = id_to_image.get(img_id)
            if not img_info:
                continue
            
            # Get source paths
            src_img_path = os.path.join(src_split_path, img_info['file_name'])
            if not os.path.exists(src_img_path):
                print(f"Image not found: {src_img_path}")
                continue
            
            # Prepare destination paths
            img_name = Path(img_info['file_name']).name
            dest_img_path = os.path.join(dest_paths[split]['images'], img_name)
            dest_label_path = os.path.join(dest_paths[split]['labels'], Path(img_name).stem + '.txt')
            
            # Copy image
            shutil.copy2(src_img_path, dest_img_path)
            
            # Convert annotation to YOLO format
            img_w = img_info['width']
            img_h = img_info['height']
            x, y, w, h = ann['bbox']
            
            # Normalize
            x_center = (x + w / 2) / img_w
            y_center = (y + h / 2) / img_h
            w /= img_w
            h /= img_h
            
            # Get class
            class_id = ann['category_id']
            class_name = categories.get(class_id, f'class_{class_id}')
            class_names.add(class_name)
            
            # Write label file
            with open(dest_label_path, 'a') as f:
                f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")
    
    # Create YAML file
    class_names = sorted(list(class_names))
    yaml_content = f"""# YOLO dataset configuration
path: {os.path.abspath(dest_root)}
train: train/images
val: valid/images
test: test/images

# Number of classes
nc: {len(class_names)}

# Class names
names: {class_names}
"""
    
    yaml_path = os.path.join(dest_root, 'dataset.yaml')
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)
    
    print(f"YOLO dataset prepared at: {os.path.abspath(dest_root)}")
    print(f"Dataset YAML file created at: {yaml_path}")


In [7]:
prepare_yolo_dataset('Data_twowheeler', 'yolo_twowheeler_dataset')

YOLO dataset prepared at: /home/STUDENTS/MSC/DS/22PD/22pd18/DL-package/yolo_twowheeler_dataset
Dataset YAML file created at: yolo_twowheeler_dataset/dataset.yaml


In [8]:
prepare_yolo_dataset('Data_helmet','yolo_data_helmet')

YOLO dataset prepared at: /home/STUDENTS/MSC/DS/22PD/22pd18/DL-package/yolo_data_helmet
Dataset YAML file created at: yolo_data_helmet/dataset.yaml
